In [ ]:
# Check Python and Spark versions
import sys
print(sys.version)
print(spark.version)

3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:38:13) [GCC 12.3.0]
3.5.1


In [2]:
import os
import subprocess
import datetime
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import functions as F
from pyspark.sql.types import *

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Get size of GCS folder
gcs_folder = 'gs://msca-bdp-data-open/final_project_reviews'

cmd = 'gsutil du -s -h ' + gcs_folder

# Execute the command for getting size
p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
for line in p.stdout.readlines():
    print(f'Total directory size: {line}')
    
retval = p.wait()

Total directory size: 75.75 GiB    gs://msca-bdp-data-open/final_project_reviews



In [ ]:
# Read data from parquet files into Spark DataFrame and show record count and schema
df_reviews = spark.read.parquet(os.path.join(gcs_folder, 'reviews_parquet'))
print(f'Records read from dataframe *reviews*: {df_reviews.count():,.0f}')
df_reviews.printSchema()
df_reviews.show(15)

Records read from dataframe *reviews*: 64,679,785
root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)

+----------+------------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+
|      asin|helpful_vote|parent_asin|rating|                text|    timestamp|               title|             user_id|verified_purchase|
+----------+------------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+
|B00HSAOFD8|           0| B00HSAOFD8|   4.0|Brushes arrived q...|1453819207000|          Four Stars|AGWL43EG6H3AO3CN2...|             true|
|B07MGKRMBG|           0

In [ ]:
# Read metadata from parquet files into Spark DataFrame and show record count and schema
df_meta = spark.read.parquet(os.path.join(gcs_folder, 'meta_parquet'))
print(f'Records read from dataframe *meta*: {df_meta.count():,.0f}')
df_meta.printSchema()
df_meta.show(15)

Records read from dataframe *meta*: 4,320,533
root
 |-- author: struct (nullable = true)
 |    |-- about: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- avatar: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- average_rating: double (nullable = true)
 |-- bought_together: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- description: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- main_category: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- price: string (nullable = true)
 |-- rating_number: long (nullable = true)
 |-- store: string (nullable = true)
 |-- subtitle: string (nullable = true)
 |-- title: string (nullable = true)



+------+--------------+---------------+--------------------+--------------------+-------------+-----------+-----+-------------+--------------------+--------+--------------------+
|author|average_rating|bought_together|          categories|         description|main_category|parent_asin|price|rating_number|               store|subtitle|               title|
+------+--------------+---------------+--------------------+--------------------+-------------+-----------+-----+-------------+--------------------+--------+--------------------+
|  NULL|           5.0|           NULL|[Beauty & Persona...|                  []|   All Beauty| B0771WJBHK| NULL|            5|        Bare Alchemy|    NULL|Soothe Serum for ...|
|  NULL|           4.0|           NULL|[Beauty & Persona...|                  []|         NULL| B09NKNNS57| NULL|           17|     DISCOVER DEVICE|    NULL|DISCOVER Tattoo B...|
|  NULL|           4.6|           NULL|[Beauty & Persona...|                  []|   All Beauty| B0C6FF5HW

In [8]:
# PROCESS DF_REVIEWS FULLY AND CLEAN IT UP
# NOTES: Filter out the verified purchases (when verified purchase is false)
# Make the time actual time (make it a timestamp)
# Drop any rows that are exact duplicates or useless for any of the necessary analysis
# DROP RATINGS THAT ARE LOWER THAN 1 AND GREATER THAN 5

from pyspark.sql import functions as F

# Start from the raw df_reviews
print("Original df_reviews row count:", df_reviews.count())

df_reviews_clean = (
    df_reviews
    # 1. Keep only verified purchases
    .filter(F.col("verified_purchase") == True)
    
    # 2. Convert timestamp (millis) -> timestamp + date
    .withColumn("review_ts", (F.col("timestamp") / 1000).cast("timestamp"))
    .withColumn("review_date", F.to_date("review_ts"))
    
    # 3. Filter ratings to valid range [1, 5] and not null
    .filter(F.col("rating").isNotNull())
    .filter((F.col("rating") >= 1.0) & (F.col("rating") <= 5.0))
    
    # 4. Remove rows with missing / empty review text
    .filter(F.col("text").isNotNull())
    .filter(F.length(F.trim(F.col("text"))) > 0)
    
    # 5. Remove rows with missing review_date
    .filter(F.col("review_date").isNotNull())
    
    # 6. Drop exact duplicate reviews
    #    (you can tweak these columns if you like)
    .dropDuplicates([
        "asin",
        "parent_asin",
        "user_id",
        "timestamp",
        "rating",
        "text"
    ])
)

print("Cleaned df_reviews row count:", df_reviews_clean.count())
df_reviews_clean.show(15, truncate=80)
df_reviews_clean.printSchema()

Original df_reviews row count: 64679785


Cleaned df_reviews row count: 59966506


+----------+------------+-----------+------+--------------------------------------------------------------------------------+-------------+------------------------------------------------------+----------------------------+-----------------+-------------------+-----------+
|      asin|helpful_vote|parent_asin|rating|                                                                            text|    timestamp|                                                 title|                     user_id|verified_purchase|          review_ts|review_date|
+----------+------------+-----------+------+--------------------------------------------------------------------------------+-------------+------------------------------------------------------+----------------------------+-----------------+-------------------+-----------+
|B00004W64Y|           9| B00004W64Y|   3.0|It adds a bit of weight and it could vibrate with a bit more &quot;oomph&quot...|1013370932000|                         Does the job, 

In [9]:
# PROCESS DF_META FULLY AND CLEAN IT UP
# NOTES: author, bought_together, subtitle description and store seem unnecessary for future analysis, drop them
# Deduplicate with title, main_category, price
# Filter rows where the price, rating_number or average rating is invalid

from pyspark.sql import functions as F
from pyspark.sql import types as T

print("Original df_meta row count:", df_meta.count())

# Drop unneeded columns and clean price / filter invalid rows
df_meta_clean = (
    df_meta
    # 1. Drop columns that are not needed for our analysis
    .drop("author", "bought_together", "subtitle", "description", "store")
    
    # 2. Convert price (string) -> numeric price_num
    .withColumn(
        "price_num",
        F.regexp_replace("price", "[$,]", "").cast(T.DoubleType())
    )
    
    # 3. Filter out rows where price, rating_number, or average_rating are invalid
    .filter(
        (F.col("price_num").isNotNull()) &
        (F.col("price_num") > 0) &
        (F.col("price_num") < 10000) &      # sanity upper bound to remove garbage
        (F.col("rating_number").isNotNull()) &
        (F.col("rating_number") > 0) &
        (F.col("average_rating").isNotNull()) &
        (F.col("average_rating") >= 1.0) &
        (F.col("average_rating") <= 5.0)
    )
    
    # 4. Drop the original string price column now that we have price_num
    .drop("price")
)

# 5. Deduplicate using title, main_category, price_num
df_meta_clean = df_meta_clean.dropDuplicates(
    ["title", "main_category", "price_num"]
)

print("Cleaned df_meta row count:", df_meta_clean.count())
df_meta_clean.printSchema()
df_meta_clean.show(15, truncate=80)

Original df_meta row count: 4320533


Cleaned df_meta row count: 1751297
root
 |-- average_rating: double (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- main_category: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating_number: long (nullable = true)
 |-- title: string (nullable = true)
 |-- price_num: double (nullable = true)



+--------------+--------------------------------------------------------------------------------+-------------------------+-----------+-------------+--------------------------------------------------------------------------------+---------+
|average_rating|                                                                      categories|            main_category|parent_asin|rating_number|                                                                           title|price_num|
+--------------+--------------------------------------------------------------------------------+-------------------------+-----------+-------------+--------------------------------------------------------------------------------+---------+
|           4.4|           [Automotive, Interior Accessories, Anti-Theft, Keyless Entry Systems]|               Automotive| B003RVQIGY|            5|                                                                                |     8.95|
|           1.0|                    

In [10]:
reviews_clean_path = "gs://msca-bdp-students-bucket/kireetij_final/reviews_clean"
meta_clean_path    = "gs://msca-bdp-students-bucket/kireetij_final/meta_clean"

# Write reviews
(
    df_reviews_clean
    .write
    .mode("overwrite")      # overwrite if folder already exists
    .parquet(reviews_clean_path)
)

# Write meta
(
    df_meta_clean
    .write
    .mode("overwrite")
    .parquet(meta_clean_path)
)

print("Saved cleaned reviews to:", reviews_clean_path)
print("Saved cleaned meta to:", meta_clean_path)

Saved cleaned reviews to: gs://msca-bdp-students-bucket/kireetij_final/reviews_clean
Saved cleaned meta to: gs://msca-bdp-students-bucket/kireetij_final/meta_clean
